# Data Preprocessing

This notebook prepares the dataset for machine learning modeling.

In [17]:
import pandas as pd
import numpy as np
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [18]:
df = pd.read_csv("../data/raw/breast_cancer.csv")

df.head()

,Age,Race,Marital Status,T Stage,N Stage,6th Stage,differentiate,Grade,A Stage,Tumor Size,Estrogen Status,Progesterone Status,Regional Node Examined,Regional Node Positive,Survival Months,Status
0,68,White,Married,T1,N1,IIA,Poorly differentiated,3,Regional,4,Positive,Positive,24,1,60,Alive
1,50,White,Married,T2,N2,IIIA,Moderately differentiated,2,Regional,35,Positive,Positive,14,5,62,Alive
2,58,White,Divorced,T3,N3,IIIC,Moderately differentiated,2,Regional,63,Positive,Positive,14,7,75,Alive
3,58,White,Married,T1,N1,IIA,Poorly differentiated,3,Regional,18,Positive,Positive,2,1,84,Alive
4,47,White,Married,T2,N1,IIB,Poorly differentiated,3,Regional,41,Positive,Positive,3,1,50,Alive


## Missing Values

In [19]:
df.isnull().sum()

Age                       0
Race                      0
Marital Status            0
T Stage                   0
N Stage                   0
6th Stage                 0
differentiate             0
Grade                     0
A Stage                   0
Tumor Size                0
Estrogen Status           0
Progesterone Status       0
Regional Node Examined    0
Regional Node Positive    0
Survival Months           0
Status                    0
dtype: int64

## Remove Unnecessary Columns

In [20]:
df = df.drop(columns=["Survival Months"])

In [21]:
df["A Stage"].value_counts()

A Stage
Regional    3932
Distant       92
Name: count, dtype: int64

In [22]:
pd.crosstab(df["6th Stage"], df["T Stage "])

T Stage,T1,T2,T3,T4
6th Stage,,,,
IIA,1305,0,0,0
IIB,0,1130,0,0
IIIA,211,428,411,0
IIIB,0,0,0,67
IIIC,87,228,122,35


In [23]:
pd.crosstab(df["6th Stage"], df["N Stage"])

N Stage,N1,N2,N3
6th Stage,,,
IIA,1305,0,0
IIB,1130,0,0
IIIA,262,788,0
IIIB,35,32,0
IIIC,0,0,472


6th Stage is derived from T Stage and N Stage and therefore introduces redundant information. That's why 6th Stage is also dropped. 

In [24]:
df = df.drop(columns=["6th Stage"])

In [25]:
pd.crosstab(df["A Stage"], df["Status"])

Status,Alive,Dead
A Stage,,
Distant,57,35
Regional,3351,581


In [26]:
pd.crosstab(df["Grade"], df["differentiate"])

differentiate,Moderately differentiated,Poorly differentiated,Undifferentiated,Well differentiated
Grade,,,,
anaplastic; Grade IV,0,0,19,0
1,0,0,0,543
2,2351,0,0,0
3,0,1111,0,0


In [27]:
df = df.drop(columns=["differentiate"])

Feature redundancy analysis revealed that:

- ```6th Stage``` was derived from combinations of ```T Stage``` and ```N Stage```.
- ```differentiate``` showed a one-to-one relationship with ```Grade```.

To reduce redundancy and simplify the feature space, these variables were removed prior to model training.

Additionally, ```Survival Months``` was excluded to prevent target leakage.

## Encode Categorical Variables

In [28]:
categorical_cols = df.select_dtypes(include="str").columns

categorical_cols

Index(['Race', 'Marital Status', 'T Stage ', 'N Stage', 'Grade', 'A Stage',
       'Estrogen Status', 'Progesterone Status', 'Status'],
      dtype='str')

In [29]:
encoders = {}

for col in categorical_cols:
    
    encoder = LabelEncoder()
    
    df[col] = encoder.fit_transform(df[col])
    
    encoders[col] = encoder

In [ ]:
df["Positive_Node_Ratio"] = (
    df["Regional Node Positive"]
    / df["Regional Node Examined"].replace(0, 1)
)

df["Tumor_Size_Group"] = pd.cut(
    df["Tumor Size"],
    bins=[0, 20, 50, 200],
    labels=["Small", "Medium", "Large"]
)

df["Hormone_Score"] = (
    (df["Estrogen Status"] == "Positive").astype(int)
    +
    (df["Progesterone Status"] == "Positive").astype(int)
)


## Train-Test Split